In [92]:
import pickle
import sys
import copy
import time

import cobra
import sympy


import multiprocessing
import multiprocessing.pool
# from multiprocessing import Process
# from threading import Thread

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params

In [2]:
# import pandas as pd
# build_files_path = '/data2/hratch/human_me/build_files/'
# human_model = cobra.io.load_json_model('/data2/hratch/human_me/input_files/toy_model.json')
# full_model = cobra.io.load_json_model('/data2/hratch/human_me/input_files/recon2_2.json')
# full_model_public = cobra.io.read_sbml_model(lp_path + 'recon2_2.xml')

# required_metabolites = pd.read_csv(build_files_path + 'required_metabolic_model_metabolites.csv', index_col = 0)

# me_model_og = copy.deepcopy(me_model)

In [3]:
from tqdm import tqdm

In [15]:
#For some reason, this protein complex used to catalyze the formation of the pre40s complex causes infeasiblity. 
#Including any of the 2 of the complex subcomponents in the precursors_fail list makes it work. Or their earlier 
#versions (up to unfolded_protein_c).

error_metabolites = ['pre40s_rrna_protein_COMPLEX_FORMATIONn_protein_complex[n]']
precurors_work = ['HGNC:21173_folded_protein[n]', 'HGNC:32790_folded_protein[n]']
precursors_fail = ['HGNC:25542_folded_protein[n]', 'HGNC:29100_folded_protein[n]']

# folded_protein[n] <-- folded_protein[c] <-- unfolded_protein[c] <-- 
# adding unfolded_protein[c] of precursors fail works too, don't need both of the precursors fail, just 1...
error_metabolites = precursors_fail.copy()

In [52]:
lp_path = '/data2/hratch/human_me/test_lp/'

def remove_metabolite(test_metabolites = [], mu_val = 0.01):
    
    with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
        me_model = pickle.load(handle)
    
    me_metabolites = [m.id for m in me_model.metabolites if 'deg_proxy' in m.id or ('mrna[n]' in m.id and 'premrna' not in m.id and 'lariats' not in m.id)]
    me_metabolites += error_metabolites

    for tm in test_metabolites:
        me_metabolites.remove(tm)
    
    ra = []
    for mm_id in me_metabolites: #me_metabolites:
        try:
            mm_obj = me_model.metabolites.get_by_id(mm_id)
        except:
            mm_obj = params.human_model.metabolites.get_by_id(mm_id)
        r = cobra.Reaction('TEST_' + mm_obj.id)
        r.add_metabolites({mm_obj: 1}, reversibly = True)
        ra.append(r)
    if len(ra) > 0:
        me_model.add_reactions(ra)
    sln, status, _ = me_model.solve_lp(mu_val = mu_val)
    return sln, status

In [617]:
sln, status = remove_metabolite()

Getting MINOS parameters...
Done in 108.978 seconds with status 0


In [239]:
with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
    me_model = pickle.load(handle)

In [ ]:
for mu_val in [0,0.01]:
    me_model.solve_lp(mu_val = mu_val)

Getting MINOS parameters...
Done in 1.3789 seconds with status 0


In [38]:
# with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
#     me_model = pickle.load(handle)
# sln, status, _ = me_model.solve_lp(mu_val = 0.01)

In [ ]:
# mrna[n] requirement: no flux through transcription elongation and transcription processing!!
# mrna_deg_proxy requirement: no flux through transcription degradation reaction
# additionally, no flux through protein degradation (polyub reaction, deubiquitination, or proteosome) reaction; 
# 2 lariat degradation reactions created